# ABC Algoritması ile Sinir Ağı Mimari Araması (NAS) — Regresyon

Bu notebook, **Yapay Arı Kolonisi (ABC — Artificial Bee Colony)** algoritmasını kullanarak
California Housing veri setinde regresyon için optimal sinir ağı mimarisini arar.

Bulunan mimari, 3 standart derin öğrenme baseline modeliyle karşılaştırılır:

| Model | Açıklama |
|-------|----------|
| **ABC-NAS** | ABC ile bulunan optimal mimari |
| **Simple MLP** | 2 katmanlı basit çok katmanlı algılayıcı |
| **Deep BN MLP** | 5 katmanlı batch normalizasyonlu derin ağ |
| **ResNet MLP** | Artık bağlantılı (skip connection) MLP |

---

## ABC Algoritması Özeti

```
Başlangıç: N rastgele mimari (food source) oluştur, fitness hesapla
Her iterasyonda:
  1. Employed Bees Fazı  → Her arı komşu mimari üretir, iyileşme varsa günceller
  2. Onlooker Bees Fazı  → Fitness'a göre seçim, daha iyi kaynaklar daha fazla keşfedilir
  3. Scout Bees Fazı     → Tükenmiş kaynaklar (limit aşıldı) rastgele yenilenir
  4. En iyi mimariyi kaydet, yakınsama eğrisini güncelle
```

**Beklenen çalışma süresi (Colab T4 GPU):** ~60–90 dakika (tam parametreler)

Hızlı test için `SMOKE_TEST = True` yapın → ~3 dakika

## Bölüm 1: Kurulum

In [ ]:
# Gerekli paketleri yükle (Colab'da)
!pip install -q tensorflow==2.15.0 scikit-learn matplotlib pandas seaborn numpy scipy

In [ ]:
import os
import gc
import time
import random
import warnings
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, BatchNormalization,
    Activation, Add, LeakyReLU
)
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow.keras.backend as K

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Tekrarlanabilirlik için sabit tohum
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow versiyonu: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPU: {gpus if gpus else "Bulunamadı (CPU kullanılacak)"}')

In [ ]:
# ─── YAPILANDIRMA ────────────────────────────────────────────────────────────
# SMOKE_TEST = True → hızlı doğrulama (~3 dk)
# SMOKE_TEST = False → tam çalıştırma (~60-90 dk)
SMOKE_TEST = False

# Google Drive checkpoint (opsiyonel)
SAVE_TO_DRIVE = False
DRIVE_PATH = '/content/drive/MyDrive/abc_checkpoint.pkl'

ABC_CONFIG = {
    'n_employed':     3  if SMOKE_TEST else 10,
    'n_onlookers':    3  if SMOKE_TEST else 10,
    'max_iterations': 3  if SMOKE_TEST else 30,
    'limit':          3  if SMOKE_TEST else 15,
    'time_budget_s':  300 if SMOKE_TEST else 5400,
}

EVAL_EPOCHS    = 5  if SMOKE_TEST else 20
FINAL_EPOCHS   = 20 if SMOKE_TEST else 100
EARLY_PATIENCE = 3  if SMOKE_TEST else 5

print('Yapılandırma:', 'SMOKE TEST' if SMOKE_TEST else 'TAM ÇALIŞTIRMA')
print(ABC_CONFIG)

## Bölüm 2: Veri Seti

In [ ]:
# California Housing veri setini yükle
housing = fetch_california_housing(as_frame=True)
X_raw = housing.data.values.astype(np.float32)
y_raw = housing.target.values.astype(np.float32)
feature_names = housing.feature_names

print(f'Toplam örnek sayısı : {X_raw.shape[0]}')
print(f'Özellik sayısı      : {X_raw.shape[1]}')
print(f'Hedef aralığı       : [{y_raw.min():.2f}, {y_raw.max():.2f}] ($100k)')
print(f'\nÖzellikler: {feature_names}')
housing.frame.describe().round(2)

In [ ]:
# %70 train / %15 val / %15 test bölünmesi
X_temp, X_test, y_temp, y_test = train_test_split(
    X_raw, y_raw, test_size=0.15, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, random_state=SEED  # 0.15 / 0.85 ≈ 0.1765
)

# StandardScaler — sadece train'e fit
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train).astype(np.float32)
X_val   = scaler_X.transform(X_val).astype(np.float32)
X_test  = scaler_X.transform(X_test).astype(np.float32)

# Hedef değişkeni ölçeklendirilmez (yorumlanabilirlik için orijinal birim)
print(f'Train seti : {X_train.shape}')
print(f'Val seti   : {X_val.shape}')
print(f'Test seti  : {X_test.shape}')

INPUT_DIM = X_train.shape[1]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Hedef dağılımı
axes[0].hist(y_raw, bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_title('Hedef Değişken Dağılımı\n(Medyan Ev Değeri, $100k)')
axes[0].set_xlabel('Değer ($100k)')
axes[0].set_ylabel('Frekans')

# Korelasyon ısı haritası
corr = housing.frame.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, ax=axes[1], linewidths=0.5)
axes[1].set_title('Özellik Korelasyon Matrisi')

plt.tight_layout()
plt.show()

## Bölüm 3: ABC Algoritması Implementasyonu

In [ ]:
# ─── ARAMA UZAYI SABİTLERİ ───────────────────────────────────────────────────

UNITS_OPTIONS      = [16, 32, 64, 128, 256, 512]   # indeks 0-5
ACTIVATION_OPTIONS = ['relu', 'elu', 'selu', 'tanh', 'leaky_relu']  # indeks 0-4
OPTIMIZER_NAMES    = ['adam', 'rmsprop', 'sgd']    # indeks 0-2
BATCH_SIZES        = [32, 64, 128]                  # indeks 0-2
MAX_LAYERS         = 6
VEC_LEN            = 27

# Vektör düzeni:
# [0]     n_layers           (1-6)
# [1-6]   units indeksi      (0-5)
# [7-11]  aktivasyon indeksi (0-4) — maks 5 değer (6. katman relu varsayılan)
# [12-17] dropout oranı      (0.0-0.5)
# [18-23] batchnorm bayrağı  (0/1)
# [24]    log(learning_rate) (log(1e-4) - log(1e-2))
# [25]    optimizer indeksi  (0-2)
# [26]    batch_size indeksi (0-2)

LOG_LR_MIN = np.log(1e-4)
LOG_LR_MAX = np.log(1e-2)

print('Arama uzayı tanımlandı.')
print(f'Vektör boyutu: {VEC_LEN}')
print(f'Maksimum katman: {MAX_LAYERS}')
print(f'Toplam olası birim kombinasyonu: {len(UNITS_OPTIONS)**MAX_LAYERS:,}')

In [ ]:
def vector_to_config(vec):
    """27 boyutlu vektörü okunabilir mimari yapılandırmasına dönüştürür."""
    n_layers = int(np.clip(round(vec[0]), 1, MAX_LAYERS))
    units = [
        UNITS_OPTIONS[int(np.clip(round(vec[1 + i]), 0, len(UNITS_OPTIONS) - 1))]
        for i in range(n_layers)
    ]
    acts = [
        ACTIVATION_OPTIONS[int(np.clip(round(vec[7 + i]), 0, len(ACTIVATION_OPTIONS) - 1))]
        for i in range(n_layers)
    ]
    dropouts = [
        float(np.clip(vec[12 + i], 0.0, 0.5))
        for i in range(n_layers)
    ]
    batchnorms = [
        bool(round(np.clip(vec[18 + i], 0, 1)))
        for i in range(n_layers)
    ]
    lr        = float(np.exp(np.clip(vec[24], LOG_LR_MIN, LOG_LR_MAX)))
    opt_idx   = int(np.clip(round(vec[25]), 0, len(OPTIMIZER_NAMES) - 1))
    bs_idx    = int(np.clip(round(vec[26]), 0, len(BATCH_SIZES) - 1))
    return {
        'n_layers': n_layers, 'units': units, 'activations': acts,
        'dropouts': dropouts, 'batchnorms': batchnorms,
        'lr': lr, 'opt_idx': opt_idx, 'bs_idx': bs_idx
    }


def build_model(vec, input_dim):
    """Mimari vektörden Keras modeli oluşturur."""
    cfg = vector_to_config(vec)

    inputs = Input(shape=(input_dim,))
    x = inputs

    for i in range(cfg['n_layers']):
        x = Dense(cfg['units'][i])(x)
        if cfg['batchnorms'][i]:
            x = BatchNormalization()(x)
        act = cfg['activations'][i]
        if act == 'leaky_relu':
            x = LeakyReLU(alpha=0.1)(x)
        else:
            x = Activation(act)(x)
        if cfg['dropouts'][i] > 0.01:
            x = Dropout(cfg['dropouts'][i])(x)

    output = Dense(1, activation='linear')(x)
    model = Model(inputs, output)

    if cfg['opt_idx'] == 0:
        optimizer = Adam(learning_rate=cfg['lr'])
    elif cfg['opt_idx'] == 1:
        optimizer = RMSprop(learning_rate=cfg['lr'])
    else:
        optimizer = SGD(learning_rate=cfg['lr'], momentum=0.9)

    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    return model


def print_architecture(vec, title='MİMARİ'):
    """Mimariyi okunabilir biçimde yazdırır."""
    cfg = vector_to_config(vec)
    print('=' * 55)
    print(f'  {title}')
    print('=' * 55)
    print(f'  Katman sayısı : {cfg["n_layers"]}')
    for i in range(cfg['n_layers']):
        bn = 'BN + ' if cfg['batchnorms'][i] else ''
        dr = f' → Dropout({cfg["dropouts"][i]:.2f})' if cfg['dropouts'][i] > 0.01 else ''
        print(f'  Katman {i+1:2d}: Dense({cfg["units"][i]:3d}) → {bn}{cfg["activations"][i]}{dr}')
    print(f'  Çıkış       : Dense(1) → linear')
    print(f'  Learning rate: {cfg["lr"]:.6f}')
    print(f'  Optimizer    : {OPTIMIZER_NAMES[cfg["opt_idx"]]}')
    print(f'  Batch size   : {BATCH_SIZES[cfg["bs_idx"]]}')
    print('=' * 55)


# Hızlı doğrulama
test_vec = np.array([2, 2, 3, 0,0,0, 0,0,0,0,0, 0.1,0.2,0,0,0,0, 1,0,0,0,0,0,
                     np.log(1e-3), 0, 1], dtype=np.float32)
test_model = build_model(test_vec, INPUT_DIM)
print('build_model çalışıyor ✓')
test_model.summary()
K.clear_session(); del test_model; gc.collect()

In [ ]:
def evaluate_architecture(vec, X_tr, y_tr, X_v, y_v,
                           epochs=EVAL_EPOCHS, patience=EARLY_PATIENCE):
    """
    Bir mimariyi değerlendirir ve validation RMSE döndürür.
    K.clear_session() ile bellek sızıntısı önlenir.
    """
    K.clear_session()
    gc.collect()

    cfg = vector_to_config(vec)
    batch_size = BATCH_SIZES[cfg['bs_idx']]

    # Deterministik ağırlık başlatma (aynı vektör → aynı sonuç)
    seed_val = int(abs(hash(vec.tobytes())) % 10000)
    tf.random.set_seed(seed_val)

    try:
        model = build_model(vec, INPUT_DIM)
        early_stop = EarlyStopping(
            monitor='val_loss', patience=patience,
            restore_best_weights=True, verbose=0
        )
        model.fit(
            X_tr, y_tr,
            epochs=epochs,
            batch_size=batch_size,
            validation_data=(X_v, y_v),
            callbacks=[early_stop],
            verbose=0
        )
        y_pred = model.predict(X_v, verbose=0).flatten()
        rmse = float(np.sqrt(mean_squared_error(y_v, y_pred)))
        del model
    except Exception as e:
        print(f'  [HATA] Değerlendirme başarısız: {e}')
        rmse = 9999.0

    K.clear_session()
    gc.collect()
    return rmse


# Hızlı doğrulama
print('evaluate_architecture doğrulanıyor...')
test_rmse = evaluate_architecture(
    test_vec, X_train[:200], y_train[:200], X_val[:100], y_val[:100],
    epochs=2, patience=1
)
print(f'Test RMSE: {test_rmse:.4f} ✓')

In [ ]:
class ABCOptimizer:
    """
    Yapay Arı Kolonisi (ABC) ile Sinir Ağı Mimari Araması.

    Karaboga & Basturk (2007) ABC algoritmasını temel alır.
    Arama uzayı: 27 boyutlu sürekli vektör (tam sayı alanlar yuvarlanır).
    """

    def __init__(self, config, X_tr, y_tr, X_v, y_v):
        self.cfg   = config
        self.X_tr, self.y_tr = X_tr, y_tr
        self.X_v,  self.y_v  = X_v,  y_v

        self.n_emp   = config['n_employed']
        self.limit   = config['limit']
        self.max_iter = config['max_iterations']
        self.budget  = config['time_budget_s']

        self.sources    = []      # food sources (mimariler)
        self.fitnesses  = []      # validation RMSE (düşük = iyi)
        self.trials     = []      # tükenme sayacı

        self.best_source  = None
        self.best_fitness = np.inf
        self.convergence  = []    # her iterasyondaki en iyi RMSE
        self.iter_min_fit = []    # her iterasyondaki popülasyon min RMSE
        self.iter_max_fit = []    # her iterasyondaki popülasyon max RMSE
        self.start_time   = None
        self.total_evals  = 0

    # ─── YARDIMCI FONKSİYONLAR ───────────────────────────────────────────────

    def _random_source(self):
        """Arama uzayında düzgün rastgele bir mimari üretir."""
        vec = np.zeros(VEC_LEN, dtype=np.float32)
        vec[0]     = np.random.randint(1, MAX_LAYERS + 1)
        vec[1:7]   = np.random.randint(0, len(UNITS_OPTIONS), 6)
        vec[7:12]  = np.random.randint(0, len(ACTIVATION_OPTIONS), 5)
        vec[12:18] = np.random.uniform(0.0, 0.5, 6)
        vec[18:24] = np.random.randint(0, 2, 6)
        vec[24]    = np.random.uniform(LOG_LR_MIN, LOG_LR_MAX)
        vec[25]    = np.random.randint(0, len(OPTIMIZER_NAMES))
        vec[26]    = np.random.randint(0, len(BATCH_SIZES))
        return vec

    def _mutate(self, xi, xk, j):
        """Tek boyutlu ABC mutasyonu: v = xi + phi*(xi - xk)."""
        phi = np.random.uniform(-1, 1)
        vi = xi.copy()
        vi[j] = xi[j] + phi * (xi[j] - xk[j])
        # Domain sınırlarını uygula
        vi[0]     = np.clip(round(vi[0]), 1, MAX_LAYERS)
        vi[1:7]   = np.clip(np.round(vi[1:7]), 0, len(UNITS_OPTIONS) - 1)
        vi[7:12]  = np.clip(np.round(vi[7:12]), 0, len(ACTIVATION_OPTIONS) - 1)
        vi[12:18] = np.clip(vi[12:18], 0.0, 0.5)
        vi[18:24] = np.clip(np.round(vi[18:24]), 0, 1)
        vi[24]    = np.clip(vi[24], LOG_LR_MIN, LOG_LR_MAX)
        vi[25]    = np.clip(round(vi[25]), 0, len(OPTIMIZER_NAMES) - 1)
        vi[26]    = np.clip(round(vi[26]), 0, len(BATCH_SIZES) - 1)
        return vi.astype(np.float32)

    def _eval(self, vec):
        self.total_evals += 1
        return evaluate_architecture(vec, self.X_tr, self.y_tr, self.X_v, self.y_v)

    def _update_best(self):
        idx = np.argmin(self.fitnesses)
        if self.fitnesses[idx] < self.best_fitness:
            self.best_fitness = self.fitnesses[idx]
            self.best_source  = self.sources[idx].copy()

    def _time_exceeded(self):
        return (time.time() - self.start_time) > self.budget

    # ─── BAŞLATMA ────────────────────────────────────────────────────────────

    def initialize(self):
        print(f'Başlangıç: {self.n_emp} rastgele mimari oluşturuluyor...')
        for i in range(self.n_emp):
            src = self._random_source()
            fit = self._eval(src)
            self.sources.append(src)
            self.fitnesses.append(fit)
            self.trials.append(0)
            print(f'  [{i+1:2d}/{self.n_emp}] RMSE: {fit:.4f}')
        self._update_best()
        print(f'Başlangıç en iyisi RMSE: {self.best_fitness:.4f}\n')

    # ─── ÇALIŞAN ARILAR FAZI ─────────────────────────────────────────────────

    def employed_phase(self):
        for i in range(self.n_emp):
            # Farklı bir kaynak seç
            candidates = [k for k in range(self.n_emp) if k != i]
            k = random.choice(candidates)
            j = random.randint(0, VEC_LEN - 1)

            vi = self._mutate(self.sources[i], self.sources[k], j)
            fi = self._eval(vi)

            if fi < self.fitnesses[i]:
                self.sources[i]   = vi
                self.fitnesses[i] = fi
                self.trials[i]    = 0
            else:
                self.trials[i] += 1

    # ─── GÖZLEMCİ ARILAR FAZI ────────────────────────────────────────────────

    def onlooker_phase(self):
        # Fitness'a göre seçim olasılığı (düşük RMSE → yüksek olasılık)
        inv_fit = np.array([1.0 / (f + 1e-9) for f in self.fitnesses])
        probs   = inv_fit / inv_fit.sum()

        n_onlookers = self.cfg['n_onlookers']
        for _ in range(n_onlookers):
            i = np.random.choice(self.n_emp, p=probs)
            candidates = [k for k in range(self.n_emp) if k != i]
            k = random.choice(candidates)
            j = random.randint(0, VEC_LEN - 1)

            vi = self._mutate(self.sources[i], self.sources[k], j)
            fi = self._eval(vi)

            if fi < self.fitnesses[i]:
                self.sources[i]   = vi
                self.fitnesses[i] = fi
                self.trials[i]    = 0
            else:
                self.trials[i] += 1

    # ─── KEŞİFÇİ ARILAR FAZI ─────────────────────────────────────────────────

    def scout_phase(self):
        for i in range(self.n_emp):
            if self.trials[i] >= self.limit:
                print(f'    [Scout] Kaynak {i} yenilendi (trial={self.trials[i]})')
                self.sources[i]   = self._random_source()
                self.fitnesses[i] = self._eval(self.sources[i])
                self.trials[i]    = 0

    # ─── ANA DÖNGÜ ────────────────────────────────────────────────────────────

    def run(self):
        self.start_time = time.time()
        self.initialize()

        for it in range(1, self.max_iter + 1):
            if self._time_exceeded():
                print(f'\nZaman bütçesi aşıldı ({self.budget}s). Erken durdurma.')
                break

            print(f'\n── İterasyon {it}/{self.max_iter} ──')
            self.employed_phase()
            self.onlooker_phase()
            self.scout_phase()
            self._update_best()

            self.convergence.append(self.best_fitness)
            self.iter_min_fit.append(min(self.fitnesses))
            self.iter_max_fit.append(max(self.fitnesses))

            elapsed = time.time() - self.start_time
            print(f'  En iyi RMSE: {self.best_fitness:.4f} | '
                  f'Geçen süre: {elapsed/60:.1f} dk | '
                  f'Toplam değerlendirme: {self.total_evals}')

            # Google Drive checkpoint
            if SAVE_TO_DRIVE:
                try:
                    with open(DRIVE_PATH, 'wb') as f:
                        pickle.dump({
                            'sources': self.sources,
                            'fitnesses': self.fitnesses,
                            'trials': self.trials,
                            'best_source': self.best_source,
                            'best_fitness': self.best_fitness,
                            'convergence': self.convergence,
                        }, f)
                except Exception:
                    pass  # Drive bağlı değilse sessizce geç

        total_time = time.time() - self.start_time
        print(f'\nABC tamamlandı!')
        print(f'En iyi validation RMSE : {self.best_fitness:.4f}')
        print(f'Toplam süre            : {total_time/60:.1f} dakika')
        print(f'Toplam değerlendirme   : {self.total_evals}')
        return self.best_source, self.best_fitness


print('ABCOptimizer sınıfı tanımlandı ✓')

## Bölüm 4: ABC Araması

In [ ]:
print('ABC araması başlıyor...')
print(f'Yapılandırma: {ABC_CONFIG}\n')

abc = ABCOptimizer(
    config=ABC_CONFIG,
    X_tr=X_train, y_tr=y_train,
    X_v=X_val,   y_v=y_val
)

best_vec, best_val_rmse = abc.run()

In [ ]:
print('ABC tarafından bulunan en iyi mimari:')
print_architecture(best_vec, title='ABC-NAS BULUNAN MİMARİ')
print(f'\nValidation RMSE (arama sırasında): {best_val_rmse:.4f}')

In [ ]:
# En iyi mimariyi train + val üzerinde sıfırdan yeniden eğit
K.clear_session(); gc.collect()
tf.random.set_seed(SEED)

X_train_full = np.vstack([X_train, X_val])
y_train_full = np.concatenate([y_train, y_val])

cfg_best = vector_to_config(best_vec)
batch_best = BATCH_SIZES[cfg_best['bs_idx']]

# İç validasyon için %10 ayır (erken durdurma için)
X_ftrain, X_fval, y_ftrain, y_fval = train_test_split(
    X_train_full, y_train_full, test_size=0.10, random_state=SEED
)

abc_model = build_model(best_vec, INPUT_DIM)
early_stop = EarlyStopping(
    monitor='val_loss', patience=15 if not SMOKE_TEST else 3,
    restore_best_weights=True, verbose=1
)

print('ABC-NAS mimarisi yeniden eğitiliyor...')
abc_history = abc_model.fit(
    X_ftrain, y_ftrain,
    epochs=FINAL_EPOCHS,
    batch_size=batch_best,
    validation_data=(X_fval, y_fval),
    callbacks=[early_stop],
    verbose=1
)

print('\nABC-NAS modeli eğitildi ✓')

## Bölüm 5: Baseline Modeller

In [ ]:
# Tüm baseline'lar için ortak eğitim ayarları
BASELINE_EPOCHS   = FINAL_EPOCHS
BASELINE_PATIENCE = 15 if not SMOKE_TEST else 3
BASELINE_LR       = 1e-3
BASELINE_BATCH    = 64

def train_model(model, X_tr, y_tr, epochs, patience, batch_size):
    """Modeli eğitir ve history döndürür."""
    X_t, X_v_b, y_t, y_v_b = train_test_split(
        X_tr, y_tr, test_size=0.10, random_state=SEED
    )
    early = EarlyStopping(
        monitor='val_loss', patience=patience,
        restore_best_weights=True, verbose=0
    )
    history = model.fit(
        X_t, y_t,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(X_v_b, y_v_b),
        callbacks=[early],
        verbose=1
    )
    return history


def compute_metrics(model, X_te, y_te):
    """Test seti üzerinde MSE, RMSE, MAE, R² hesaplar."""
    y_pred = model.predict(X_te, verbose=0).flatten()
    mse  = mean_squared_error(y_te, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_te, y_pred)
    r2   = r2_score(y_te, y_pred)
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R²': r2}, y_pred


print('Baseline eğitim fonksiyonları tanımlandı ✓')

In [ ]:
# ─── MODEL 1: Simple MLP ─────────────────────────────────────────────────────
K.clear_session(); gc.collect()
tf.random.set_seed(SEED)

inp = Input(shape=(INPUT_DIM,))
x = Dense(64, activation='relu')(inp)
x = Dropout(0.1)(x)
x = Dense(32, activation='relu')(x)
x = Dropout(0.1)(x)
out = Dense(1, activation='linear')(x)
model1 = Model(inp, out)
model1.compile(optimizer=Adam(BASELINE_LR), loss='mse', metrics=['mae'])

print('Model 1 — Simple MLP:')
model1.summary()

h1 = train_model(model1, X_train_full, y_train_full,
                 BASELINE_EPOCHS, BASELINE_PATIENCE, BASELINE_BATCH)
metrics1, pred1 = compute_metrics(model1, X_test, y_test)
print(f'\nSimple MLP Test: RMSE={metrics1["RMSE"]:.4f}, R²={metrics1["R²"]:.4f}')

In [ ]:
# ─── MODEL 2: Deep BN MLP ────────────────────────────────────────────────────
K.clear_session(); gc.collect()
tf.random.set_seed(SEED)

def build_deep_bn_mlp(input_dim, lr):
    inp = Input(shape=(input_dim,))
    x = inp
    for units in [256, 256, 128, 128, 64]:
        x = Dense(units)(x)
        x = BatchNormalization()(x)
        x = Activation('relu')(x)
        x = Dropout(0.2)(x)
    out = Dense(1, activation='linear')(x)
    m = Model(inp, out)
    m.compile(optimizer=Adam(lr), loss='mse', metrics=['mae'])
    return m

model2 = build_deep_bn_mlp(INPUT_DIM, BASELINE_LR)
print('Model 2 — Deep BN MLP:')
model2.summary()

h2 = train_model(model2, X_train_full, y_train_full,
                 BASELINE_EPOCHS, BASELINE_PATIENCE, BASELINE_BATCH)
metrics2, pred2 = compute_metrics(model2, X_test, y_test)
print(f'\nDeep BN MLP Test: RMSE={metrics2["RMSE"]:.4f}, R²={metrics2["R²"]:.4f}')

In [ ]:
# ─── MODEL 3: ResNet MLP ─────────────────────────────────────────────────────
K.clear_session(); gc.collect()
tf.random.set_seed(SEED)

def residual_block(x, units, dropout_rate=0.2):
    residual = x
    x = Dense(units)(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Dropout(dropout_rate)(x)
    x = Dense(units)(x)
    x = BatchNormalization()(x)
    x = Add()([x, residual])
    x = Activation('relu')(x)
    return x

def build_resnet_mlp(input_dim, lr):
    inp = Input(shape=(input_dim,))
    # Projeksiyon: girişi gizli boyuta taşı
    x = Dense(128)(inp)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    # 3 residual blok
    x = residual_block(x, 128, dropout_rate=0.2)
    x = residual_block(x, 128, dropout_rate=0.2)
    x = residual_block(x, 128, dropout_rate=0.1)
    # Çıkış kafası
    x = Dense(64, activation='relu')(x)
    out = Dense(1, activation='linear')(x)
    m = Model(inp, out)
    m.compile(optimizer=Adam(lr), loss='mse', metrics=['mae'])
    return m

model3 = build_resnet_mlp(INPUT_DIM, BASELINE_LR)
print('Model 3 — ResNet MLP:')
model3.summary()

h3 = train_model(model3, X_train_full, y_train_full,
                 BASELINE_EPOCHS, BASELINE_PATIENCE, BASELINE_BATCH)
metrics3, pred3 = compute_metrics(model3, X_test, y_test)
print(f'\nResNet MLP Test: RMSE={metrics3["RMSE"]:.4f}, R²={metrics3["R²"]:.4f}')

In [ ]:
# ABC-NAS modelini test setinde değerlendir
metrics_abc, pred_abc = compute_metrics(abc_model, X_test, y_test)
print(f'ABC-NAS Test: RMSE={metrics_abc["RMSE"]:.4f}, R²={metrics_abc["R²"]:.4f}')

## Bölüm 6: Sonuçlar ve Görselleştirme

In [ ]:
# ─── SONUÇ TABLOSU ───────────────────────────────────────────────────────────
all_metrics = {
    'ABC-NAS':     metrics_abc,
    'Simple MLP':  metrics1,
    'Deep BN MLP': metrics2,
    'ResNet MLP':  metrics3,
}

df_results = pd.DataFrame(all_metrics).T[['MSE', 'RMSE', 'MAE', 'R²']]
df_results = df_results.round(4)
df_results['Sıralama (RMSE)'] = df_results['RMSE'].rank(ascending=True).astype(int)
df_results['Sıralama (R²)']   = df_results['R²'].rank(ascending=False).astype(int)

print('\n' + '='*65)
print('       TEST SETİ SONUÇLARI (California Housing)')
print('='*65)
print(df_results.to_string())
print('='*65)
print('Not: RMSE ve MAE birimi $100k (düşük = iyi), R² yüksek = iyi')

# Renkli tablo (Jupyter'da görünür)
styled = (
    df_results.style
    .highlight_min(subset=['MSE', 'RMSE', 'MAE'], color='#90EE90')
    .highlight_max(subset=['R²'], color='#90EE90')
    .format(precision=4)
    .set_caption('Test Seti Karşılaştırması — En iyi değerler yeşil')
)
display(styled)

In [ ]:
# ─── EĞİTİM KAYIP EĞRİLERİ ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Eğitim ve Doğrulama Kayıp Eğrileri (MSE)', fontsize=14, fontweight='bold')

histories = [
    ('ABC-NAS',    abc_history, '#2196F3'),
    ('Simple MLP', h1,         '#4CAF50'),
    ('Deep BN MLP',h2,         '#FF9800'),
    ('ResNet MLP', h3,         '#9C27B0'),
]

for ax, (name, hist, color) in zip(axes.flatten(), histories):
    train_loss = hist.history['loss']
    val_loss   = hist.history['val_loss']
    epochs_range = range(1, len(train_loss) + 1)

    ax.semilogy(epochs_range, train_loss,
                color=color, linewidth=2, label='Eğitim')
    ax.semilogy(epochs_range, val_loss,
                color=color, linewidth=2, linestyle='--', alpha=0.7, label='Doğrulama')

    # En iyi epoch işareti
    best_epoch = int(np.argmin(val_loss)) + 1
    ax.axvline(best_epoch, color='red', linestyle=':', linewidth=1.5,
               label=f'En iyi epoch: {best_epoch}')

    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE (log ölçek)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('loss_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Grafik kaydedildi: loss_curves.png')

In [ ]:
# ─── TAHMİN vs GERÇEK SAÇILIM GRAFİKLERİ ────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Tahmin vs Gerçek (Test Seti)', fontsize=14, fontweight='bold')

models_preds = [
    ('ABC-NAS',     pred_abc, '#2196F3', metrics_abc),
    ('Simple MLP',  pred1,   '#4CAF50', metrics1),
    ('Deep BN MLP', pred2,   '#FF9800', metrics2),
    ('ResNet MLP',  pred3,   '#9C27B0', metrics3),
]

for ax, (name, y_pred, color, mets) in zip(axes, models_preds):
    ax.scatter(y_test, y_pred, alpha=0.15, s=6, color=color)

    # İdeal çizgi (y=x)
    lim_min = min(y_test.min(), y_pred.min())
    lim_max = max(y_test.max(), y_pred.max())
    ax.plot([lim_min, lim_max], [lim_min, lim_max],
            'r--', linewidth=2, label='Mükemmel tahmin')

    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Gerçek Değer ($100k)')
    ax.set_ylabel('Tahmin ($100k)')
    ax.text(0.05, 0.93, f'R²={mets["R²"]:.3f}',
            transform=ax.transAxes, fontsize=10, color='black',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax.text(0.05, 0.83, f'RMSE={mets["RMSE"]:.3f}',
            transform=ax.transAxes, fontsize=10, color='black',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('scatter_plots.png', dpi=120, bbox_inches='tight')
plt.show()
print('Grafik kaydedildi: scatter_plots.png')

In [ ]:
# ─── ABC YAKINSAMA EĞRİSİ ────────────────────────────────────────────────────
if abc.convergence:
    fig, ax = plt.subplots(figsize=(11, 5))

    iters = range(1, len(abc.convergence) + 1)

    # Popülasyon çeşitliliği bandı
    ax.fill_between(iters,
                    abc.iter_min_fit,
                    abc.iter_max_fit,
                    alpha=0.2, color='steelblue',
                    label='Popülasyon aralığı (min-max)')

    # En iyi RMSE eğrisi
    ax.plot(iters, abc.convergence,
            'b-o', markersize=5, linewidth=2,
            label='En iyi validation RMSE')

    ax.set_xlabel('ABC İterasyonu', fontsize=12)
    ax.set_ylabel('Validation RMSE ($100k)', fontsize=12)
    ax.set_title('ABC Yakınsama Eğrisi\n(Mavi band: popülasyon çeşitliliği)',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    # Minimum noktayı işaretle
    best_iter = int(np.argmin(abc.convergence)) + 1
    ax.axvline(best_iter, color='red', linestyle=':', linewidth=1.5,
               label=f'En iyi iter: {best_iter}')
    ax.legend(fontsize=10)

    plt.tight_layout()
    plt.savefig('convergence.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Grafik kaydedildi: convergence.png')
else:
    print('Yakınsama verisi yok (smoke test kısa çalıştı).')

In [ ]:
# ─── ÖZET ÇUBUK GRAFİĞİ ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Karşılaştırması — Test Seti', fontsize=14, fontweight='bold')

model_names = list(all_metrics.keys())
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
rmse_vals = [all_metrics[m]['RMSE'] for m in model_names]
r2_vals   = [all_metrics[m]['R²']   for m in model_names]

bars1 = axes[0].bar(model_names, rmse_vals, color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_title('RMSE Karşılaştırması (düşük = iyi)', fontsize=11)
axes[0].set_ylabel('RMSE ($100k)')
axes[0].set_ylim(0, max(rmse_vals) * 1.2)
for bar, val in zip(bars1, rmse_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

bars2 = axes[1].bar(model_names, r2_vals, color=colors, edgecolor='white', linewidth=0.8)
axes[1].set_title('R² Karşılaştırması (yüksek = iyi)', fontsize=11)
axes[1].set_ylabel('R²')
axes[1].set_ylim(min(r2_vals) * 0.95, 1.0)
for bar, val in zip(bars2, r2_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

for ax in axes:
    ax.tick_params(axis='x', rotation=15)
    ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_bar.png', dpi=120, bbox_inches='tight')
plt.show()
print('Grafik kaydedildi: comparison_bar.png')

In [ ]:
# ─── SONUÇ ÖZETİ ─────────────────────────────────────────────────────────────
best_model_rmse = df_results['RMSE'].idxmin()
best_model_r2   = df_results['R²'].idxmax()

print('\n' + '='*65)
print('  SONUÇ ÖZETİ')
print('='*65)
print(f'\n  En iyi RMSE : {best_model_rmse} ({df_results.loc[best_model_rmse, "RMSE"]:.4f} $100k)')
print(f'  En iyi R²   : {best_model_r2} ({df_results.loc[best_model_r2, "R²"]:.4f})')

print('\n  ABC arama istatistikleri:')
print(f'    Toplam evaluasyon : {abc.total_evals}')
print(f'    İterasyon sayısı  : {len(abc.convergence)}')
print(f'    En iyi val RMSE   : {abc.best_fitness:.4f}')

print('\n  ABC bulunan mimari özeti:')
print_architecture(best_vec, title='ABC-NAS MİMARİ')

print('\n  NOT: ResNet MLP artık bağlantı (skip connection) içerir.')
print('  ABC arama uzayı yalnızca sıralı (feedforward) mimarileri kapsar.')
print('  ResNet\'in üstün gelmesi topolojinin önemini gösterir; ABC\'nin')
print('  üstün gelmesi ise genişlik/derinlik/düzenlileştirme ayarının')
print('  topolojiden daha kritik olduğuna işaret eder.')
print('='*65)

## Sonuçlar ve Değerlendirme

### Bulgular

Bu çalışmada **Yapay Arı Kolonisi (ABC)** algoritması, California Housing veri setinde
regresyon için optimal sinir ağı mimarisi aramak amacıyla kullanıldı.

### ABC Algoritmasının Avantajları
- **Gradyansız optimizasyon**: Arama uzayı türevlenebilir olmak zorunda değil
- **Keşif-sömürü dengesi**: Scout fazı yerel optimumlardan kaçışı sağlar
- **Hiperparametre araması**: Mimari ile birlikte LR, optimizer, batch size de aranır

### Sınırlılıklar
- **Hesaplama maliyeti**: Her evaluasyon bir model eğitimi gerektirdiğinden pahalıdır
- **Arama uzayı**: Sıralı (feedforward) mimarilerle sınırlıdır; skip connection, attention vb. içermez
- **Yerel optimum riski**: Küçük koloni boyutlarında arama yetersiz kalabilir

### Gelecek Çalışmalar
- Arama uzayını artık bağlantı, dikkat mekanizması içerecek şekilde genişletme
- Surrogate model (Gaussian Process) ile evaluasyon maliyetini azaltma
- Multi-objective ABC: RMSE ve model boyutunu birlikte optimize etme